# Multi-Modal Supply Chain Network Analytics & Cascading Delay Prediction
## Notebook 1: Library Imports & Initial Raw Data Loading
This cell extracts raw multi-modal logistics datasets from the project `data/` directory, inspects DataFrame shapes, column types `.info()`, and null value matrices.

In [ ]:
import os
import sys
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 10

# Resolve data paths
script_dir = os.getcwd()
data_dir = os.path.join(script_dir, 'data') if os.path.exists(os.path.join(script_dir, 'data')) else os.path.join(os.path.dirname(script_dir), 'data')

sc_path = os.path.join(data_dir, 'DataCoSupplyChainDataset.csv')
tel_path = os.path.join(data_dir, 'cascading_logistics_telemetry.csv')
log_path = os.path.join(data_dir, 'tokenized_access_logs.csv')
desc_path = os.path.join(data_dir, 'DescriptionDataCoSupplyChain.csv')

print("=== CELL 1: INITIAL DATA LOADING & METADATA SUMMARY ===")

# 1. Supply Chain Dataset
df_sc_raw = pd.read_csv(sc_path, encoding='latin-1')
print(f"\n1. Supply Chain Dataset Shape: {df_sc_raw.shape[0]:,} rows x {df_sc_raw.shape[1]} columns")
print("--- Top Null Column Counts (Supply Chain) ---")
print(df_sc_raw.isnull().sum()[df_sc_raw.isnull().sum() > 0].head(10))

# 2. Logistics Telemetry Dataset
df_tel_raw = pd.read_csv(tel_path)
print(f"\n2. Logistics Telemetry Dataset Shape: {df_tel_raw.shape[0]:,} rows x {df_tel_raw.shape[1]} columns")
print("--- Null Summary (Telemetry) ---")
print(df_tel_raw.isnull().sum())

# 3. Tokenized Access Logs Dataset
df_log_raw = pd.read_csv(log_path)
print(f"\n3. Access Logs Dataset Shape: {df_log_raw.shape[0]:,} rows x {df_log_raw.shape[1]} columns")
print("--- Null Summary (Access Logs) ---")
print(df_log_raw.isnull().sum())

# 4. Metadata Description
if os.path.exists(desc_path):
    df_desc = pd.read_csv(desc_path)
    print(f"\n4. Description Metadata Shape: {df_desc.shape[0]} fields documented")

## Notebook 2: Data Cleaning & Pre-processing Operations
Applies strict data cleaning rules across all 4 datasets:
1. **Supply Chain:** `snake_case` column standardization, purging null critical IDs (`order_id`, `customer_id`), deduplicating exact transaction rows, calculating `delay_margin` ($Days_{real} - Days_{scheduled}$), and standardizing ISO-8601 UTC dates.
2. **Telemetry:** Filtering negative dwell times, purging out-of-bounds GPS coordinates ($|lat| > 90$ or $|lon| > 180$), modal weather imputation, and purging rapid RFID scanner duplicates (< 5s window).
3. **Access Logs:** Deduplicating rapid repeated pings (< 5s window for same IP/URL) and standardizing hourly date tags.

In [ ]:
print("=== CELL 2: DATA CLEANING & PRE-PROCESSING AUDIT ===")

def to_snake(col_name):
    s = re.sub(r'[\(\)]', '', col_name.strip())
    s = re.sub(r'[\s\-_]+', '_', s)
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s)
    return s.lower()

# 1. Supply Chain Cleaning
df_sc = df_sc_raw.copy()
df_sc.columns = [to_snake(c) for c in df_sc.columns]
if 'order_customer_id' in df_sc.columns and 'customer_id' not in df_sc.columns:
    df_sc['customer_id'] = df_sc['order_customer_id']

sc_raw_len = len(df_sc)
df_sc = df_sc[df_sc['order_id'].notna() & df_sc['customer_id'].notna()].copy()
null_id_purged = sc_raw_len - len(df_sc)

sc_dupes = df_sc.duplicated().sum()
df_sc = df_sc.drop_duplicates().reset_index(drop=True)

df_sc['delay_margin'] = df_sc['days_for_shipping_real'] - df_sc['days_for_shipment_scheduled']
df_sc['order_date_iso'] = pd.to_datetime(df_sc['order_date_date_orders'], errors='coerce', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')
df_sc['shipping_date_iso'] = pd.to_datetime(df_sc['shipping_date_date_orders'], errors='coerce', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

print(f"Supply Chain Audit: Raw={sc_raw_len:,} -> Clean={len(df_sc):,} | Purged Null IDs={null_id_purged} | Purged Dupes={sc_dupes}")

# 2. Telemetry Cleaning
df_tel = df_tel_raw.copy()
tel_raw_len = len(df_tel)
dwell_col = 'actual_dwell' if 'actual_dwell' in df_tel.columns else 'dwell_duration_seconds'
neg_dwell = (df_tel[dwell_col] < 0).sum()
df_tel = df_tel[df_tel[dwell_col] >= 0].copy().reset_index(drop=True)
df_tel['dwell_time_mins'] = np.round(df_tel[dwell_col] / 60.0, 2)

lat_col = 'gps_latitude' if 'gps_latitude' in df_tel.columns else 'gps_lat'
lon_col = 'gps_longitude' if 'gps_longitude' in df_tel.columns else 'gps_lon'
bad_gps = ((df_tel[lat_col] > 90) | (df_tel[lat_col] < -90) | (df_tel[lon_col] > 180) | (df_tel[lon_col] < -180)).sum()
df_tel = df_tel[~((df_tel[lat_col] > 90) | (df_tel[lat_col] < -90) | (df_tel[lon_col] > 180) | (df_tel[lon_col] < -180))].copy().reset_index(drop=True)

weather_nulls = df_tel['weather_condition'].isna().sum()
fac_col = 'current_facility' if 'current_facility' in df_tel.columns else 'facility_id'
weather_mode = df_tel['weather_condition'].mode()[0] if not df_tel['weather_condition'].dropna().empty else 'CLEAR'
df_tel['weather_condition'] = df_tel['weather_condition'].fillna(weather_mode)
df_tel['event_timestamp_iso'] = pd.to_datetime(df_tel['event_timestamp'], errors='coerce', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

df_tel['parsed_ts'] = pd.to_datetime(df_tel['event_timestamp'], errors='coerce', utc=True)
df_tel = df_tel.sort_values(by=['shipment_id', 'parsed_ts']).reset_index(drop=True)
time_delta = df_tel.groupby('shipment_id')['parsed_ts'].diff().dt.total_seconds()
same_fac = df_tel[fac_col] == df_tel.groupby('shipment_id')[fac_col].shift(1)
tel_dupes = ((time_delta < 5.0) & same_fac).sum()
df_tel = df_tel[~((time_delta < 5.0) & same_fac)].copy().reset_index(drop=True)

print(f"Telemetry Audit: Raw={tel_raw_len:,} -> Clean={len(df_tel):,} | Neg Dwell Purged={neg_dwell} | Bad GPS Purged={bad_gps} | Imputed Weather={weather_nulls} | Purged Scanner Dupes={tel_dupes}")

# 3. Access Logs Cleaning
df_log = df_log_raw.copy()
log_raw_len = len(df_log)
df_log['parsed_ts'] = pd.to_datetime(df_log['Date'], errors='coerce', utc=True)
df_log = df_log.sort_values(by=['ip', 'url', 'parsed_ts']).reset_index(drop=True)
log_delta = df_log.groupby(['ip', 'url'])['parsed_ts'].diff().dt.total_seconds()
log_dupes = (log_delta < 5.0).sum()
df_log = df_log[~(log_delta < 5.0)].copy().reset_index(drop=True)
df_log['date_iso'] = df_log['parsed_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')
df_log['hour_clean'] = df_log['parsed_ts'].dt.hour

print(f"Access Logs Audit: Raw={log_raw_len:,} -> Clean={len(df_log):,} | Purged Rapid Access Dupes={log_dupes}")

## Notebook 3: Exploratory Data Analysis (EDA) & Visualizations
Generates key domain visual proofs:
1. **Chart 1:** Distribution of Late Delivery Risk (`late_delivery_risk`) across shipping modes.
2. **Chart 2:** Delay Margin Distribution ($Days_{real} - Days_{scheduled}$).
3. **Chart 3:** Facility Congestion Heatmap derived from telemetry data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Distribution of Late Delivery Risk across Shipping Modes
sns.countplot(data=df_sc, x='shipping_mode', hue='late_delivery_risk', palette='viridis', ax=axes[0])
axes[0].set_title('Late Delivery Risk by Shipping Mode', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Shipping Mode')
axes[0].set_ylabel('Transaction Volume')
axes[0].legend(title='Late Risk', labels=['On-Time (0)', 'Late Risk (1)'])

# Chart 2: Delay Margin Distribution
sns.histplot(data=df_sc, x='delay_margin', bins=15, kde=True, color='teal', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='SLA Scheduled Delivery')
axes[1].set_title('Delay Margin Distribution (Actual - Scheduled Days)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delay Margin (Days)')
axes[1].set_ylabel('Order Count')
axes[1].legend()

# Chart 3: Facility Congestion Heatmap
df_tel['capacity_utilization'] = np.round(df_tel['yard_queue_count'] / df_tel['yard_max_capacity'].replace(0, 1), 4)
congestion_pivot = df_tel.groupby([fac_col, 'weather_condition'])['capacity_utilization'].mean().unstack().fillna(0)

sns.heatmap(congestion_pivot, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Mean Utilization'}, ax=axes[2])
axes[2].set_title('Facility Congestion Heatmap by Weather', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Weather Condition')
axes[2].set_ylabel('Logistics Facility Hub')

plt.tight_layout()
plt.show()

## Notebook 4: Predictive Delay Modeling & Metric Evaluation
Trains a Random Forest Classifier to predict late delivery risk (`late_delivery_risk`) and exports evaluation metrics: Precision, Recall, F1-Score, and Mean Absolute Error (MAE).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, mean_absolute_error, classification_report

print("=== CELL 4: PREDICTIVE DELAY MODELING (RANDOM FOREST) ===")

feature_cols = [
    'days_for_shipment_scheduled', 'order_item_quantity', 'sales',
    'order_item_discount_rate', 'order_item_profit_ratio', 'product_price'
]
X = df_sc[feature_cols].fillna(0)
y = df_sc['late_delivery_risk'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- Machine Learning Model Evaluation Metrics ---")
print(f" * Precision Score : {precision * 100:.2f}%")
print(f" * Recall Score    : {recall * 100:.2f}%")
print(f" * F1-Score        : {f1 * 100:.2f}%")
print(f" * Mean Abs Error  : {mae:.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

## Notebook 5: Cleaned Data Export
Writes all final cleaned DataFrames directly into `cleaned_data/` as `cleaned_supply_chain.csv`, `cleaned_telemetry.csv`, and `cleaned_access_logs.csv`.

In [ ]:
print("=== CELL 5: CLEANED DATA EXPORT ===")

out_dir_1 = os.path.join(os.getcwd(), 'cleaned_data')
out_dir_2 = os.path.join(os.getcwd(), 'server', 'src', 'cleaned_data')

for d in [out_dir_1, out_dir_2]:
    os.makedirs(d, exist_ok=True)
    df_sc.to_csv(os.path.join(d, 'cleaned_supply_chain.csv'), index=False)
    df_tel.to_csv(os.path.join(d, 'cleaned_telemetry.csv'), index=False)
    df_log.to_csv(os.path.join(d, 'cleaned_access_logs.csv'), index=False)
    print(f" -> Successfully wrote clean CSVs to {d}")